# SFT, two epochs

One-axis mutation from the one-epoch SFT. English is the weakest script at 25% DR@5.

In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path

def run(cmd, **kw):
    print("$", " ".join(str(c) for c in cmd), flush=True)
    return subprocess.run(cmd, check=True, **kw)

def find_repo():
    hits=[p for p in Path("/kaggle/input").rglob("rule_checker.py") if p.exists() and (p.parents[2]/"scripts"/"train.py").exists()]
    exact=[p for p in hits if "datasets/rockerleo/htd-code" in str(p)]
    hits=exact or hits; hits.sort(key=lambda p: len(p.parts))
    if not hits: raise SystemExit("code not mounted")
    return hits[0].parents[2]

WORK=Path("/kaggle/working/htd")
if WORK.exists(): shutil.rmtree(WORK)
shutil.copytree(find_repo(), WORK); os.chdir(WORK); sys.path.insert(0,str(WORK/"src"))
for p in Path("/kaggle/input").rglob("hf_token.txt"):
    os.environ.setdefault("HF_TOKEN", p.read_text().strip())
    os.environ.setdefault("HUGGINGFACE_API_KEY", p.read_text().strip())
assert os.environ.get("HF_TOKEN"), "attach htd-secrets"
FIRE="CohereLabs/tiny-aya-fire"
assert "--method" in (WORK/"scripts"/"train.py").read_text()
print("code ok")

def assert_adapter(name):
    p=WORK/"outputs"/name
    assert (p/"adapter_config.json").exists(), name
    assert (p/"adapter_model.safetensors").stat().st_size > 1_000_000, name
    print(name, "adapter ok")

In [ ]:
pairs = next(Path("/kaggle/input").rglob("sft_train.jsonl")).parent
dst = WORK / "data" / "generated" / "pairs"
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(pairs, dst)
assert (dst / "sft_train.jsonl").exists()
print("pairs:", sorted(x.name for x in dst.iterdir()))

In [ ]:
run([sys.executable, "scripts/train.py", "--method", "sft", "--base", FIRE,
     "--data-dir", "data/generated/pairs", "--output", "outputs/sft2_lr1e4",
     "--epochs", "2", "--lr", "1e-4", "--rank", "16", "--alpha", "32",
     "--batch", "1", "--grad-accum", "8"])
assert_adapter("sft2_lr1e4")